## LightGBM Test

In [23]:
%load_ext autoreload
%autoreload 2

In [24]:
%%bash
# autoreload


if ! command -v pdftohtml &> /dev/null
then
    echo "pdftohtml is not installed. Installing now..."
    sudo apt install pdftohtml
else
    echo "pdftohtml is already installed."
fi

pdftohtml is already installed.


We use Poppler to convert PDFs to XMLs. To work with Poppler in Python, we have created `PdfFeatures` module, which can be found in `pdf_features/PdfFeatures.py`.

In [25]:
from pdf_features.PdfFeatures import PdfFeatures

To open a PDF file with PdfFeatures module, simply write:

In [ ]:
# pdf_features: PdfFeatures = PdfFeatures.from_labeled_data("pdf-labeled-data", "test_data", "1001.4384.pdf_page_4")

When you open `pdf_features` like this, the XML file is saved in a temporary path and handled on the fly.

If you want to save the XML file, you should provide a path where it can be saved:

In [54]:
# pdf-labeled-data/labeled_data/token_type/test_data
import os
from tqdm import tqdm
all_files = os.listdir("pdf-labeled-data/labeled_data/token_type/validation_data")
all_files = [f for f in all_files if 'DS_Store' not in f]
pdfs_features = []
for file in tqdm(all_files):
    pdf_feature = PdfFeatures.from_labeled_data("pdf-labeled-data", "validation_data", file)
    pdfs_features.append(pdf_feature)


100%|██████████| 2074/2074 [00:34<00:00, 60.95it/s] 


In [ ]:
import sys
import os

features_path = os.path.join(os.getcwd(), "features")
src_path = os.path.join(os.getcwd(), "src")

if features_path not in sys.path:
    sys.path.append(features_path)
    sys.path.append(src_path)


from src.adapters.ml.pdf_tokens_type_trainer.ModelConfiguration import ModelConfiguration
from src.adapters.ml.pdf_tokens_type_trainer.TokenTypeTrainer import TokenTypeTrainer

def get_pdf_features_labels() -> PdfFeatures:
    # Assuming that you are loading your own labels in this part.
    # I'm just going to put a list with a single file for demonstration.
    pdf_features: PdfFeatures = PdfFeatures.from_pdf_path("test_pdfs/regular.pdf")
    labeled_pdf_features_list: list[PdfFeatures] = [pdf_features]
    return labeled_pdf_features_list

model_configuration = ModelConfiguration()
labeled_pdf_features_list: list[PdfFeatures] = pdfs_features
trainer = TokenTypeTrainer(labeled_pdf_features_list, model_configuration)
train_labels = [token.token_type.get_index() for token in trainer.loop_tokens()]
import time
start_time = time.time()
trainer.predict("models/token_type_example_model.model")
end_time = time.time()
print(f"Prediction took {end_time - start_time} seconds, average per page: {(end_time - start_time)/len(pdfs_features)} seconds")

# The feature extraction takes 70 seconds or so, so the prediction is actually ~166.73 s, average per page: ~0.0804 seconds

100%|██████████| 2074/2074 [00:00<00:00, 7550.38it/s]

Prediction took 236.73961091041565 seconds, average per page: 0.11414638905998827 seconds


In [58]:
predictions = [token.prediction for token in trainer.loop_tokens()]

In [59]:
correct = [1 for label, pred in zip(train_labels, predictions) if label == pred]
accuracy = sum(correct) / len(train_labels)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 87.46%


In [62]:
from dataloader import DocLayNetDatasetSegmented
dataset = DocLayNetDatasetSegmented(split='validation')

pdfs_features = []
for i in tqdm(range(2000)):
    pdfs_features.append(dataset[i])
dataset.save_cache("data_cache_segmenter_valid.pkl")

print([tok.prediction for tok in pdfs_features[0].pages[0].tokens])
features_path = os.path.join(os.getcwd(), "features")
src_path = os.path.join(os.getcwd(), "src")

if features_path not in sys.path:
    sys.path.append(features_path)
    sys.path.append(src_path)

100%|██████████| 2000/2000 [04:07<00:00,  8.08it/s]


[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]


In [63]:
from src.adapters.ml.fast_trainer.ParagraphExtractorTrainer import ParagraphExtractorTrainer

model_configuration = ModelConfiguration()
labeled_pdf_features_list: list[PdfFeatures] = pdfs_features
trainer = ParagraphExtractorTrainer(labeled_pdf_features_list, model_configuration)
train_labels = [token.prediction for token in trainer.loop_tokens()]

start_time = time.time()
trainer.predict("models/paragraph_extractor_example_model_.model")
end_time = time.time()
print(f"Prediction took {end_time - start_time} seconds, average per page: {(end_time - start_time)/len(pdfs_features)} seconds")

100%|██████████| 2000/2000 [00:00<00:00, 8278.76it/s] 

Prediction took 99.50238609313965 seconds, average per page: 0.049751193046569823 seconds


In [64]:
predictions = [token.prediction for token in trainer.loop_tokens()]
correct = [1 for label, pred in zip(train_labels, predictions) if label == pred]
accuracy = sum(correct) / len(train_labels)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 98.84%
